In [1]:
!pip install lightgbm xgboost imbalanced-learn -q

In [2]:
# ================================================================
# SAML-D AML Detection — Complete Final Version
# FL  : FedAvg via SGDClassifier (log_loss) — weights average cleanly
# ML  : XGBoost + LightGBM per bank
# MoE : Static | Performance | Typology-Aware (novel)
#
# SETUP (run first cell):
#   !pip install lightgbm xgboost imbalanced-learn -q
#
# Dataset: berkanoztas/synthetic-transaction-monitoring-dataset-aml
# ================================================================

import os, gc, time, warnings, random
from collections import defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("Device: CPU (no PyTorch — sklearn SGD FedAvg)")

T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush():   gc.collect()

# ================================================================
# CONFIG
# ================================================================
SUBSAMPLE  = 9_500_000   
N_BANKS    = 4
FL_ROUNDS  = 30
LR         = 0.01
TEST_FRAC  = 0.15
VAL_FRAC   = 0.15
MIN_FRAUD  = 2
THRESH     = 0.5
AP_KS      = (50, 100, 200)
ALPHAS     = [0.5, 0.1, 0.05]

# ================================================================
# 1. DATA
# ================================================================
def find_csv():
    candidates = [
        '/kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
        '/kaggle/input/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
    ]
    for p in candidates:
        if os.path.exists(p): return p
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv') and 'SAML' in f.upper():
                return os.path.join(root, f)
    return None


def load_data():
    path = find_csv()
    if not path:
        raise FileNotFoundError("SAML-D not found. Add dataset to notebook.")
    print(f"Loading: {path}")
    df = pd.read_csv(path, nrows=SUBSAMPLE, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"Loaded {len(df):,} rows | "
          f"fraud: {df['Is_laundering'].sum():,} "
          f"({df['Is_laundering'].mean()*100:.3f}%)")
    return df


HIGH_RISK = {'mexico', 'turkey', 'morocco', 'uae', 'iran',
             'nigeria', 'russia', 'venezuela', 'north korea'}


def preprocess(df):
    df   = df.copy()
    y    = df['Is_laundering'].astype(int).values
    tArr = df['Laundering_type'].astype(str).fillna('Unknown').values
    lArr = df['Sender_bank_location'].astype(str).fillna('Unknown').values

    print(f"Laundering: {y.sum():,} ({y.mean()*100:.3f}%)")
    print(f"Typologies ({len(np.unique(tArr))}): {sorted(np.unique(tArr))}")

    le_t = LabelEncoder(); df['typ_enc'] = le_t.fit_transform(tArr)
    le_s = LabelEncoder(); df['sl_enc']  = le_s.fit_transform(lArr)
    le_r = LabelEncoder()
    df['rl_enc'] = le_r.fit_transform(
        df['Receiver_bank_location'].astype(str).fillna('Unknown'))

    amt = df['Amount'].fillna(0).abs()
    df['amt_log']    = np.log1p(amt)
    df['amt_round']  = (amt % 1000 == 0).astype(int)
    df['amt_thresh'] = (amt < 10000).astype(int)

    t = pd.to_numeric(df['Time'], errors='coerce').fillna(0)
    df['hr_sin'] = np.sin(2 * np.pi * t / 86400)
    df['hr_cos'] = np.cos(2 * np.pi * t / 86400)

    le_p = LabelEncoder()
    df['pt_enc'] = le_p.fit_transform(
        df['Payment_type'].astype(str).fillna('Unknown'))

    df['sl_risk'] = df['Sender_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK)))
    df['rl_risk'] = df['Receiver_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK)))
    df['cross_border'] = (
        df['Sender_bank_location'].astype(str) !=
        df['Receiver_bank_location'].astype(str)).astype(int)
    df['curr_mm'] = (
        df['Payment_currency'].astype(str) !=
        df['Received_currency'].astype(str)).astype(int)

    feats = ['amt_log', 'amt_round', 'amt_thresh',
             'hr_sin', 'hr_cos', 'pt_enc', 'typ_enc',
             'sl_enc', 'rl_enc', 'sl_risk', 'rl_risk',
             'cross_border', 'curr_mm']
    X = df[feats].fillna(0).values.astype(np.float32)
    print(f"X: {X.shape}")
    return X, y, tArr, lArr


# ================================================================
# 2. PARTITIONING
# ================================================================
def partition(X, y, typ_arr, alpha):
    """
    Dirichlet partition with guaranteed MIN_FRAUD per bank.
    Fraud redistributed if any bank falls below threshold.
    """
    np.random.seed(42)
    fi = np.where(y == 1)[0]
    li = np.where(y == 0)[0]
    np.random.shuffle(fi)
    np.random.shuffle(li)

    props = np.random.dirichlet([alpha] * N_BANKS)
    fcnts = (props * len(fi)).astype(int)
    fcnts[-1] = len(fi) - fcnts[:-1].sum()
    fcnts = np.maximum(fcnts, 0)

    # Guarantee minimum fraud per bank
    for i in range(N_BANKS):
        if fcnts[i] < MIN_FRAUD:
            deficit = MIN_FRAUD - fcnts[i]
            donor   = np.argmax(fcnts)
            if fcnts[donor] > MIN_FRAUD + deficit:
                fcnts[donor] -= deficit
                fcnts[i]     += deficit

    lper  = len(li) // N_BANKS
    banks = []; fp = 0; lp = 0

    for i in range(N_BANKS):
        ln  = lper if i < N_BANKS - 1 else len(li) - lp
        idx = np.concatenate([fi[fp:fp+fcnts[i]], li[lp:lp+ln]])
        np.random.shuffle(idx)
        Xi, yi, ti = X[idx], y[idx], typ_arr[idx]
        banks.append(_split(i, Xi, yi, ti))
        fp += fcnts[i]; lp += ln

    print(f"\nPartition α={alpha} — {len(banks)} banks:")
    for b in banks:
        top = list(b['typ_dist'].keys())[:2]
        print(f"  Bank {b['id']}: {b['n_total']:7d} txns | "
              f"{b['n_fraud']:4d} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"top: {top}")
    return banks


def _split(bid, X, y, typ):
    strat = y if y.sum() >= 2 else None
    try:
        Xtr,Xte,ytr,yte,ttr,tte = train_test_split(
            X, y, typ, test_size=TEST_FRAC,
            random_state=42, stratify=strat)
        s2 = ytr if ytr.sum() >= 2 else None
        Xtr,Xvl,ytr,yvl,ttr,tvl = train_test_split(
            Xtr, ytr, ttr,
            test_size=VAL_FRAC / (1 - TEST_FRAC),
            random_state=42, stratify=s2)
    except Exception:
        s = max(2, len(X) // 5)
        Xtr,ytr,ttr = X[s:],    y[s:],    typ[s:]
        Xvl,yvl,tvl = X[s//2:s],y[s//2:s],typ[s//2:s]
        Xte,yte,tte = X[:s//2], y[:s//2], typ[:s//2]

    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)

    return dict(
        id=bid,
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,   typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y),
        fraud_frac=y.mean(),
        typ_dist=pd.Series(typ).value_counts(normalize=True).to_dict(),
    )


def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k   = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y


# ================================================================
# 3. FEDAVG via SGDClassifier
# ================================================================
def make_fl_model(y_all=None):
    """
    SGDClassifier with log_loss = logistic regression via SGD.
    class_weight passed as dict (not 'balanced') for partial_fit compatibility.
    """
    if y_all is not None and len(np.unique(y_all)) == 2:
        cw = compute_class_weight('balanced',
                                   classes=np.array([0, 1]), y=y_all)
        sw = {0: float(cw[0]), 1: float(cw[1])}
    else:
        sw = {0: 1.0, 1: 1000.0}   # ~0.1% fraud rate fallback

    return SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-4,
        learning_rate='constant',
        eta0=LR,
        max_iter=1,
        warm_start=True,
        class_weight=sw,
        random_state=42,
        tol=None,
    )


def get_weights(model):
    return model.coef_.copy(), model.intercept_.copy()


def set_weights(model, coef, intercept):
    model.coef_      = coef.copy()
    model.intercept_ = intercept.copy()


def fedavg_aggregate(weight_list, sizes):
    total    = sum(sizes)
    avg_coef = sum((s/total)*c for (c,_), s in zip(weight_list, sizes))
    avg_int  = sum((s/total)*i for (_,i), s in zip(weight_list, sizes))
    return avg_coef, avg_int


def run_fedavg(banks, input_dim):
    print(f"\n[FedAvg — SGD LogReg] {FL_ROUNDS} rounds × {len(banks)} banks")
    CLASSES = np.array([0, 1])

    # Initialise global model
    all_X      = np.vstack([b['X_train'][:200] for b in banks])
    all_y      = np.concatenate([b['y_train'][:200] for b in banks])
    all_y_full = np.concatenate([b['y_train'] for b in banks])
    if 0 not in all_y: all_y[0] = 0
    if 1 not in all_y: all_y[1] = 1

    global_model = make_fl_model(y_all=all_y_full)
    global_model.partial_fit(all_X, all_y, classes=CLASSES)

    history = []

    for rnd in range(1, FL_ROUNDS + 1):
        local_weights = []
        local_sizes   = []

        for bank in banks:
            Xtr, ytr = bank['X_train'], bank['y_train']
            Xtr_s, ytr_s = safe_smote(Xtr, ytr)

            # Clone global weights to local model
            local = make_fl_model(y_all=ytr)
            # Initialise local model architecture
            init_X = Xtr_s[:min(10, len(Xtr_s))]
            init_y = ytr_s[:min(10, len(ytr_s))]
            if 0 not in init_y: init_y[0] = 0
            if 1 not in init_y: init_y[-1] = 1
            local.partial_fit(init_X, init_y, classes=CLASSES)
            # Copy global weights in
            gc_l, ic_l = get_weights(global_model)
            set_weights(local, gc_l, ic_l)

            # Local SGD — 3 passes
            idx = np.random.permutation(len(Xtr_s))
            for _ in range(3):
                local.partial_fit(Xtr_s[idx], ytr_s[idx], classes=CLASSES)

            local_weights.append(get_weights(local))
            local_sizes.append(len(Xtr))
            del local

        # FedAvg aggregation
        avg_c, avg_i = fedavg_aggregate(local_weights, local_sizes)
        set_weights(global_model, avg_c, avg_i)

        if rnd % 5 == 0 or rnd == FL_ROUNDS:
            print(f"  Round {rnd}/{FL_ROUNDS} | {elapsed()}", end='  ')
            f1s = []
            for bank in banks:
                yt = bank['y_test']
                if yt.sum() == 0: continue
                probs = global_model.predict_proba(bank['X_test'])[:, 1]
                preds = (probs >= THRESH).astype(int)
                f1s.append(f1_score(yt, preds, zero_division=0))
                history.append(dict(
                    round=rnd,
                    bank_id=bank['id'],
                    f1=f1_score(yt, preds, zero_division=0),
                    auprc=average_precision_score(yt, probs),
                    recall=recall_score(yt, preds, zero_division=0),
                    precision=precision_score(yt, preds, zero_division=0),
                ))
            print(f"mean F1={np.mean(f1s):.4f}")
        gc.collect()

    return global_model, history


# ================================================================
# 4. LOCAL EXPERTS
# ================================================================
class XGBExpert:
    name = 'xgb'
    def __init__(self, pw):
        self.m = xgb.XGBClassifier(
            n_estimators=200, max_depth=6,
            scale_pos_weight=pw,
            use_label_encoder=False,
            eval_metric='aucpr',
            tree_method='hist',
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42, verbosity=0)
        self.trained = False

    def fit(self, X, y):
        self.m.fit(X, y)
        self.trained = True

    def predict_proba(self, X):
        return self.m.predict_proba(X)[:, 1] if self.trained \
               else np.full(len(X), 0.5)


class LGBMExpert:
    name = 'lgbm'
    def __init__(self, pw):
        self.m = lgb.LGBMClassifier(
            n_estimators=200, max_depth=6,
            scale_pos_weight=pw,
            subsample=0.8,
            colsample_bytree=0.8,
            device='cpu',
            random_state=42, verbose=-1)
        self.trained = False

    def fit(self, X, y):
        self.m.fit(X, y)
        self.trained = True

    def predict_proba(self, X):
        return self.m.predict_proba(X)[:, 1] if self.trained \
               else np.full(len(X), 0.5)


def train_experts(banks):
    experts = {}
    for bank in banks:
        bid = bank['id']
        experts[bid] = {}
        Xtr, ytr = safe_smote(bank['X_train'], bank['y_train'])
        n0 = (ytr==0).sum(); n1 = max((ytr==1).sum(), 1)
        pw = float(n0 / n1)

        for Cls in [XGBExpert, LGBMExpert]:
            m = Cls(pw)
            m.fit(Xtr, ytr)
            probs = m.predict_proba(bank['X_test'])
            preds = (probs >= THRESH).astype(int)
            f1  = f1_score(bank['y_test'], preds, zero_division=0)
            ap  = average_precision_score(bank['y_test'], probs)
            rec = recall_score(bank['y_test'], preds, zero_division=0)
            print(f"  Bank {bid} {m.name.upper():5s}: "
                  f"F1={f1:.4f}  AUPRC={ap:.4f}  Recall={rec:.4f}")
            experts[bid][m.name] = m

    return experts


# ================================================================
# 5. MoE GATES
# ================================================================
class StaticGate:
    name = 'moe_static'

    def get_weights(self, n, avail, **kw):
        w = np.where(avail, 1., 0.)
        return w / w.sum() if w.sum() > 0 else w

    def update(self, *a, **k):
        pass


class PerformanceGate:
    name = 'moe_performance'

    def __init__(self):
        self._w = {}

    def update(self, bid, val_f1s, avail):
        s = np.where(avail, np.clip(val_f1s, 0, None), 0.)
        self._w[bid] = s / s.sum() if s.sum() > 0 else np.ones(len(s)) / len(s)

    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)


class TypologyAwareGate:
    """
    NOVEL CONTRIBUTION:
    Routes each transaction to the expert with the highest historical F1
    on that specific laundering typology (from validation set).
    Falls back to performance-weighted if typology unseen.
    """
    name = 'moe_typology_aware'

    def __init__(self):
        self._tbl = defaultdict(lambda: defaultdict(dict))
        self._fb  = {}

    def update(self, bid, typ_f1s, avail, val_f1s):
        for ei, (tf, av) in enumerate(zip(typ_f1s, avail)):
            if av:
                self._tbl[bid][ei] = tf
        s = np.where(avail, np.clip(val_f1s, 0, None), 0.)
        self._fb[bid] = s / s.sum() if s.sum() > 0 \
                        else np.ones(len(s)) / len(s)

    def get_weights(self, n, avail, bid=None, txn_typ=None, **kw):
        if txn_typ is None or bid not in self._tbl:
            return self._fb.get(bid, np.ones(n) / n)
        w = np.zeros(n)
        for ei in range(n):
            if avail[ei] and ei in self._tbl[bid]:
                w[ei] = self._tbl[bid][ei].get(txn_typ, 0.)
        return w / w.sum() if w.sum() > 0 \
               else self._fb.get(bid, np.ones(n) / n)


def _typ_f1(y_val, probs_val, typ_val):
    preds = (probs_val >= THRESH).astype(int)
    out   = {}
    for t in np.unique(typ_val):
        mask = typ_val == t
        if mask.sum() < 2 or y_val[mask].sum() == 0: continue
        out[t] = f1_score(y_val[mask], preds[mask], zero_division=0)
    return out


def fl_probs(model, X):
    try:
        return model.predict_proba(X)[:, 1]
    except Exception:
        return np.full(len(X), 0.5)


# ================================================================
# 6. METRICS
# ================================================================
TYP_W = {
    'Cycle': 2.5, 'Bipartite': 2.5, 'Stacked Bipartite': 2.5,
    'Structuring': 2.5, 'Smurfing': 2.0,
    'Scatter-Gather': 2.0, 'Gather-Scatter': 2.0,
    'Fan_Out': 2.0, 'Fan_In': 2.0,
    'Layered_Fan_Out': 2.0, 'Layered_Fan_In': 2.0,
    'Deposit-Send': 2.0, 'Over-Invoicing': 2.0,
    'Single_large': 1.5, 'Cash_Withdrawal': 1.5,
    'Behavioural_Change_1': 1.5, 'Behavioural_Change_2': 1.5,
}


def compute_metrics(y_true, probs, typ=None):
    if y_true.sum() == 0: return _em()
    preds = (probs >= THRESH).astype(int)
    f1    = f1_score(y_true, preds, zero_division=0)
    prec  = precision_score(y_true, preds, zero_division=0)
    rec   = recall_score(y_true, preds, zero_division=0)
    try:    auc_roc = roc_auc_score(y_true, probs)
    except: auc_roc = 0.5
    try:    auprc = average_precision_score(y_true, probs)
    except: auprc = float(y_true.mean())
    mcc  = matthews_corrcoef(y_true, preds) if preds.sum() > 0 else 0.
    b2   = 4; d = b2*prec + rec
    f2   = (1+b2) * prec * rec / d if d > 0 else 0.
    try:
        tn,fp,fn,tp = confusion_matrix(y_true, preds, labels=[0,1]).ravel()
        gmean = np.sqrt(tp/max(tp+fn,1) * tn/max(tn+fp,1))
    except: gmean = 0.
    bal  = balanced_accuracy_score(y_true, preds)
    sidx = np.argsort(probs)[::-1]
    apk  = {f'ap_at_{k}': float(y_true[sidx[:min(k,len(y_true))]].sum() /
                                  max(min(k,len(y_true)), 1))
            for k in AP_KS}
    typ_cov = 0.; typ_wf1 = 0.
    if typ is not None:
        laund = [t for t in np.unique(typ)
                 if 'Normal' not in t and t != 'Unknown']
        if laund:
            typ_cov = sum(
                1 for t in laund
                if ((typ==t) & (y_true==1) & (preds==1)).sum() > 0
            ) / len(laund)
        ws = wt = 0.
        for t in np.unique(typ):
            mask = typ == t
            if mask.sum() < 2 or y_true[mask].sum() == 0: continue
            f = f1_score(y_true[mask], preds[mask], zero_division=0)
            w = TYP_W.get(t, 1.5 if 'Normal' not in t else 0.)
            ws += w*f; wt += w
        typ_wf1 = ws / max(wt, 1e-8)
    return {
        'f1': f1, 'precision': prec, 'recall': rec, 'auc_roc': auc_roc,
        'auprc': auprc, 'mcc': mcc, 'f2': f2, 'gmean': gmean, 'bal_acc': bal,
        **apk, 'typ_coverage': typ_cov, 'typ_wf1': typ_wf1
    }


def _em():
    return {k: 0. for k in [
        'f1','precision','recall','auc_roc','auprc','mcc','f2',
        'gmean','bal_acc','ap_at_50','ap_at_100','ap_at_200',
        'typ_coverage','typ_wf1']}


def agg(ml):
    if not ml: return {}
    return {k: round(float(np.mean([m[k] for m in ml])), 4)
            for k in ml[0]}


def fairness(bank_f1s, local_f1s=None):
    r = dict(
        client_equity = round(float(np.std(bank_f1s)), 4),
        worst_bank_f1 = round(float(min(bank_f1s)), 4),
        best_bank_f1  = round(float(max(bank_f1s)), 4),
    )
    if local_f1s and len(local_f1s) == len(bank_f1s):
        r['collab_gain'] = round(
            float(np.mean([a-b for a,b in zip(bank_f1s, local_f1s)])), 4)
    return r


# ================================================================
# 7. FULL EVALUATION
# ================================================================
def evaluate_all(banks, fl_model, experts, alpha):
    rows  = []
    n_exp = 3   # fedavg + xgb + lgbm

    # Local-only best F1 per bank (for collaboration gain metric)
    loc_f1 = {}
    for bank in banks:
        bid = bank['id']; yt = bank['y_test']
        if yt.sum() == 0: continue
        best = 0.
        for en in ['xgb', 'lgbm']:
            m = experts[bid].get(en)
            if m and m.trained:
                p = m.predict_proba(bank['X_test'])
                best = max(best, f1_score(yt, (p>=THRESH).astype(int),
                                          zero_division=0))
        loc_f1[bid] = best

    def _row(strat, mtype, bm_list, lf=None):
        a  = agg(bm_list)
        fa = fairness([m['f1'] for m in bm_list], lf)
        return {'strategy': strat, 'alpha': alpha,
                'model_type': mtype, **a, **fa}

    # ── FedAvg ───────────────────────────────────────────────
    bm = []; lf = []
    for bank in banks:
        yt = bank['y_test']; tte = bank.get('typ_test')
        if yt.sum() == 0: continue
        bm.append(compute_metrics(yt, fl_probs(fl_model, bank['X_test']), tte))
        lf.append(loc_f1.get(bank['id'], 0))
    rows.append(_row('fedavg', 'fl', bm, lf))

    # ── XGB + LGBM ───────────────────────────────────────────
    for en in ['xgb', 'lgbm']:
        bm = []
        for bank in banks:
            yt = bank['y_test']; tte = bank.get('typ_test')
            if yt.sum() == 0: continue
            m = experts[bank['id']].get(en)
            probs = m.predict_proba(bank['X_test']) if (m and m.trained) \
                    else np.full(len(yt), 0.5)
            bm.append(compute_metrics(yt, probs, tte))
        rows.append(_row(en, 'local_expert', bm))

    # ── 3 MoE gates ──────────────────────────────────────────
    for gate in [StaticGate(), PerformanceGate(), TypologyAwareGate()]:
        bm = []; lf = []
        for bank in banks:
            bid = bank['id']; yt = bank['y_test']
            tte = bank.get('typ_test', np.array(['Unknown']*len(yt)))
            tvl = bank.get('typ_val',  np.array(['Unknown']*len(bank['y_val'])))
            if yt.sum() == 0: continue

            tps=[]; vf1s=[]; avl=[]; tyf=[]

            # FedAvg expert
            pv = fl_probs(fl_model, bank['X_val'])
            pt = fl_probs(fl_model, bank['X_test'])
            vf1s.append(f1_score(bank['y_val'],
                                  (pv>=THRESH).astype(int), zero_division=0))
            tyf.append(_typ_f1(bank['y_val'], pv, tvl))
            avl.append(True); tps.append(pt)

            # XGB + LGBM experts
            for en in ['xgb', 'lgbm']:
                m = experts[bid].get(en)
                if m and m.trained:
                    pv = m.predict_proba(bank['X_val'])
                    pt = m.predict_proba(bank['X_test'])
                    vf1s.append(f1_score(bank['y_val'],
                                          (pv>=THRESH).astype(int), zero_division=0))
                    tyf.append(_typ_f1(bank['y_val'], pv, tvl))
                    avl.append(True); tps.append(pt)
                else:
                    vf1s.append(0.); tyf.append({})
                    avl.append(False)
                    tps.append(np.full(len(yt), 0.5))

            avl  = np.array(avl)
            vf1s = np.array(vf1s)

            # Gate weights + aggregate
            if isinstance(gate, TypologyAwareGate):
                gate.update(bid, tyf, avl, val_f1s=vf1s)
                ap = np.zeros(len(yt))
                for ut in np.unique(tte):
                    mask = tte == ut
                    w = gate.get_weights(n_exp, avl, bid=bid, txn_typ=ut)
                    for i, pt in enumerate(tps):
                        ap[mask] += w[i] * pt[mask]
            elif isinstance(gate, PerformanceGate):
                gate.update(bid, vf1s, avl)
                w  = gate.get_weights(n_exp, avl, bid=bid)
                ap = sum(w[i]*tps[i] for i in range(n_exp))
            else:
                w  = gate.get_weights(n_exp, avl)
                ap = sum(w[i]*tps[i] for i in range(n_exp))

            bm.append(compute_metrics(yt, ap, tte))
            lf.append(loc_f1.get(bid, 0))

        rows.append(_row(gate.name, 'moe', bm, lf))

    return rows


# ================================================================
# 8. PLOTS
# ================================================================
COLORS = {
    'fedavg':             '#4F46A0',
    'xgb':                '#A32D2D',
    'lgbm':               '#185FA5',
    'moe_static':         '#AAAAAA',
    'moe_performance':    '#1D9E75',
    'moe_typology_aware': '#9333EA',
}


def plot_alpha(df_bm, fl_hist, alpha):
    strats = sorted(df_bm['strategy'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f'SAML-D AML — Dirichlet α={alpha}\n'
        f'FedAvg (SGD-LogReg) + XGB + LGBM + 3 MoE Gates',
        fontsize=11, fontweight='bold')

    # ── Primary metrics bar ──────────────────────────────────
    ax = axes[0]; x = np.arange(len(strats)); w = 0.22
    for mi, (m, lbl, c) in enumerate([
            ('auprc', 'AUPRC*', '#2E4057'),
            ('f2',    'F2*',    '#048A81'),
            ('f1',    'F1',     '#54C6EB')]):
        vals = [float(df_bm[df_bm['strategy']==s][m].mean())
                for s in strats]
        ax.bar(x + (mi-1)*w, vals, w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],
                       fontsize=8, rotation=40, ha='right')
    ax.set_ylim(0, 1); ax.legend(fontsize=8, frameon=False)
    ax.set_title('Primary Metrics  (* key at 0.1% imbalance)',
                 fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)

    # ── AUPRC vs Recall ──────────────────────────────────────
    ax = axes[1]
    for si, s in enumerate(strats):
        c  = COLORS.get(s, '#999')
        ap = float(df_bm[df_bm['strategy']==s]['auprc'].mean())
        rc = float(df_bm[df_bm['strategy']==s]['recall'].mean())
        ax.bar(si - 0.18, ap, 0.35, color=c, alpha=0.85)
        ax.bar(si + 0.18, rc, 0.35, color=c, alpha=0.4,
               edgecolor=c, linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],
                       fontsize=8, rotation=40, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title('AUPRC (solid) vs Recall (faded)',
                 fontsize=9, fontweight='bold')
    ax.legend(handles=[Patch(color='grey', alpha=0.85, label='AUPRC'),
                       Patch(color='grey', alpha=0.4,  label='Recall')],
              fontsize=8, frameon=False)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)

    # ── FedAvg learning curve ────────────────────────────────
    ax = axes[2]
    if fl_hist:
        dfh = pd.DataFrame(fl_hist)
        g   = dfh.groupby('round')[['f1','auprc','recall']].mean().reset_index()
        ax.plot(g['round'], g['auprc'],  label='AUPRC',
                color='#1D9E75', lw=2, marker='s', ms=4)
        ax.plot(g['round'], g['f1'],     label='F1',
                color='#4F46A0', lw=2, marker='o', ms=4)
        ax.plot(g['round'], g['recall'], label='Recall',
                color='#D85A30', lw=2, marker='^', ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0, 1)
    ax.set_title('FedAvg Learning Curve (mean across banks)',
                 fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(alpha=0.25)

    plt.tight_layout()
    p = f'{OUT}/results_alpha{str(alpha).replace(".","")}.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")


def plot_heatmap(df_all):
    metrics = ['auprc','mcc','f2','f1','recall',
               'ap_at_100','typ_coverage','typ_wf1']
    col_lbl = ['AUPRC*','MCC*','F2*','F1','Recall',
               'AP@100','Typ.Cov','Typ.WF1']
    alphas  = sorted(df_all['alpha'].unique())
    strats  = sorted(df_all['strategy'].unique())

    fig, axes = plt.subplots(1, len(alphas),
                              figsize=(5*len(alphas)+1, 5), sharey=True)
    if len(alphas) == 1: axes = [axes]
    fig.suptitle('Full Benchmark Heatmap  (* = PRIMARY metrics, purple border)',
                 fontsize=11, fontweight='bold', y=1.01)

    for ax, alpha in zip(axes, alphas):
        sub = df_all[df_all['alpha'] == alpha]
        mat = np.zeros((len(strats), len(metrics)))
        for si, s in enumerate(strats):
            row = sub[sub['strategy'] == s]
            for mi, m in enumerate(metrics):
                if not row.empty and m in row.columns:
                    v = row[m].values[0]
                    mat[si, mi] = float(v) if pd.notna(v) else 0.

        im = ax.imshow(mat, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
        ax.set_xticks(range(len(col_lbl)))
        ax.set_xticklabels(col_lbl, fontsize=8, rotation=35, ha='right')
        ax.set_yticks(range(len(strats)))
        ax.set_yticklabels([s.replace('moe_','MoE-') for s in strats], fontsize=8)
        ax.set_title(f'α = {alpha}', fontsize=10, fontweight='bold')

        for i in range(len(strats)):
            for j in range(len(col_lbl)):
                v = mat[i, j]
                ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                        fontsize=7, color='white' if v > 0.65 else 'black')
        for pc in [0, 1, 2]:
            for i in range(len(strats)):
                ax.add_patch(plt.Rectangle((pc-.5, i-.5), 1, 1,
                             fill=False, edgecolor='#9333EA', lw=1.5))

    plt.tight_layout()
    p = f'{OUT}/benchmark_heatmap.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")


def print_summary(df_all):
    print("\n" + "="*72)
    print("FINAL BENCHMARK — SAML-D AML Detection")
    print("PRIMARY: AUPRC, MCC, F2  |  AUC-ROC = reference only")
    print("="*72)
    cols  = ['strategy','alpha','auprc','mcc','f2','f1','recall',
             'ap_at_100','typ_coverage','typ_wf1',
             'client_equity','worst_bank_f1','collab_gain']
    avail = [c for c in cols if c in df_all.columns]
    print(df_all[avail].sort_values(['alpha','auprc'],
                                    ascending=[True,False]).to_string(index=False))
    print("\nBEST per α (by AUPRC):")
    for alpha in sorted(df_all['alpha'].unique()):
        sub  = df_all[df_all['alpha'] == alpha]
        best = sub.loc[sub['auprc'].idxmax()]
        print(f"  α={alpha}: {best['strategy']:25s}  "
              f"AUPRC={best['auprc']:.4f}  "
              f"F2={best['f2']:.4f}  "
              f"Recall={best['recall']:.4f}  "
              f"MCC={best['mcc']:.4f}")


# ================================================================
# MAIN
# ================================================================
if __name__ == '__main__':
    print(f"\n{'='*55}")
    print("SAML-D — FedAvg + XGB + LGBM + 3 MoE Gates")
    print(f"{'='*55}\n")

    df_raw = load_data()
    X, y, typ_arr, loc_arr = preprocess(df_raw)
    del df_raw; flush()
    print(f"\nData ready | {elapsed()}")

    all_bm   = []
    all_hist = []

    for alpha in ALPHAS:
        print(f"\n{'='*55}")
        print(f"Partition: Dirichlet α={alpha}")
        print(f"{'='*55}")

        banks = partition(X, y, typ_arr, alpha)

        fl_model, fl_hist = run_fedavg(banks, X.shape[1])
        for h in fl_hist: h['alpha'] = alpha
        all_hist.extend(fl_hist)
        flush()

        print(f"\nLocal experts (XGB + LGBM)...")
        experts = train_experts(banks)
        flush()

        print(f"\nEvaluating all models + MoE gates...")
        bm_rows = evaluate_all(banks, fl_model, experts, alpha)
        all_bm.extend(bm_rows)

        plot_alpha(pd.DataFrame(bm_rows), fl_hist, alpha)
        print(f"α={alpha} done | {elapsed()}")
        flush()

    df_bm = pd.DataFrame(all_bm)
    df_bm.to_csv(f'{OUT}/saml_benchmark.csv', index=False)
    pd.DataFrame(all_hist).to_csv(f'{OUT}/saml_fl_history.csv', index=False)

    print_summary(df_bm)
    plot_heatmap(df_bm)

    print(f"\nTotal runtime: {elapsed()}")
    print(f"\nOutputs in {OUT}:")
    for f in sorted(os.listdir(OUT)):
        if f.endswith(('.csv', '.png')):
            print(f"  {f}")

Device: CPU (no PyTorch — sklearn SGD FedAvg)

SAML-D — FedAvg + XGB + LGBM + 3 MoE Gates

Loading: /kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv
Loaded 9,500,000 rows | fraud: 9,869 (0.104%)
Laundering: 9,869 (0.104%)
Typologies (28): ['Behavioural_Change_1', 'Behavioural_Change_2', 'Bipartite', 'Cash_Withdrawal', 'Cycle', 'Deposit-Send', 'Fan_In', 'Fan_Out', 'Gather-Scatter', 'Layered_Fan_In', 'Layered_Fan_Out', 'Normal_Cash_Deposits', 'Normal_Cash_Withdrawal', 'Normal_Fan_In', 'Normal_Fan_Out', 'Normal_Foward', 'Normal_Group', 'Normal_Mutual', 'Normal_Periodical', 'Normal_Plus_Mutual', 'Normal_Small_Fan_Out', 'Normal_single_large', 'Over-Invoicing', 'Scatter-Gather', 'Single_large', 'Smurfing', 'Stacked Bipartite', 'Structuring']
X: (9500000, 13)

Data ready | 1.3m

Partition: Dirichlet α=0.5

Partition α=0.5 — 4 banks:
  Bank 0: 2374558 txns | 2026 fraud (0.085%) | top: ['Normal_Small_Fan_Out', 'Normal_Fan_Out']
  Bank 1: 2377414 txns | 